# Whisper vs faster-whisper vs Vosk

Using a T4 GPU to run this

In [ ]:
# install packages and vosk model
!pip install -q openai-whisper faster-whisper vosk gtts pydub jiwer
!wget -q -nc https://alphacephei.com/vosk/models/vosk-model-small-en-us-0.15.zip
!unzip -q -o vosk-model-small-en-us-0.15.zip
print("installed stuff")

In [ ]:
# generate test data
import os
from gtts import gTTS
from pydub import AudioSegment
import re

texts = [
    ("The quick brown fox jumps over the lazy dog near the riverbank. This is a short test sentence for the transcription engine."),
    
    ("Artificial intelligence is transforming the way we work and live. By automating repetitive tasks, "
     "machines allow humans to focus on creative problem solving and strategic thinking. Voice assistants "
     "are now embedded in our phones, cars, and homes."),
     
    ("The development of artificial intelligence has been one of the most significant "
     "technological achievements of the twenty first century. Machine learning algorithms "
     "can now process vast amounts of data and identify patterns that would be impossible "
     "for humans to detect. Natural language processing has enabled computers to understand "
     "and generate human language with remarkable accuracy. Speech recognition systems "
     "like Whisper have made it possible to transcribe audio in real time. "),
     
    ("Space exploration has always fascinated humankind, pushing the boundaries of what is possible. "
     "From the early days of the Apollo missions to the modern rovers exploring the surface of Mars, "
     "we continually strive to understand the universe around us. Telescopes like Hubble and James Webb "
     "have captured breathtaking images of distant galaxies and nebulae, revealing a cosmos far more "
     "complex and beautiful than anyone could have imagined. As private companies join national space agencies "
     "in the pursuit of commercial spaceflight and lunar habitats, the next decade promises unprecedented "
     "advancements in our journey to become an interplanetary species, unlocking the secrets of the solar system."),
     
    ("Climate change remains one of the greatest global challenges of our time, requiring immediate "
     "and coordinated action from governments, corporations, and individuals alike. The scientific consensus "
     "is clear that the Earth's average temperature is rising due to greenhouse gas emissions from human activities "
     "such as burning fossil fuels and deforestation. This warming trend has profound impacts on weather patterns, "
     "leading to more frequent and severe heatwaves, droughts, and powerful storms. Melting polar ice caps "
     "and thermal expansion are causing sea levels to rise, threatening coastal communities worldwide. "
     "Transitioning to renewable energy sources like solar, wind, and hydroelectric power is crucial to mitigate "
     "these effects. Furthermore, adopting sustainable agricultural practices and investing in green technologies "
     "will help build a resilient future for generations to come. The window to act is narrowing, making innovation "
     "and international cooperation more vital than ever.")
]

test_files = []
print("generating audio...")

for idx, raw_text in enumerate(texts):
    ground_truth = re.sub(r'[^\w\s]', '', raw_text).lower()
    
    mp3_path = f"test_audio_{idx+1}.mp3"
    wav_path = f"test_audio_{idx+1}.wav"
    
    tts = gTTS(text=raw_text, lang="en")
    tts.save(mp3_path)
    
    audio = AudioSegment.from_mp3(mp3_path)
    audio = audio.set_frame_rate(16000).set_channels(1)
    duration_secs = len(audio) / 1000.0
    audio.export(wav_path, format="wav")
    
    test_files.append({
        'id': idx + 1,
        'wav_path': wav_path,
        'duration': duration_secs,
        'ground_truth': ground_truth
    })
    print(f"test {idx+1}: {duration_secs:.1f}s")


In [ ]:
# testing openai whisper
import whisper
import time
import torch
from jiwer import wer
import re

model_name = "small"
print(f"\n--- openai whisper ({model_name}) ---")

t0 = time.time()
model_openai = whisper.load_model(model_name)
load_time_ow = time.time() - t0
print(f"load time: {load_time_ow:.2f}s")

results_ow = []

for test in test_files:
    print(f"\nrunning file {test['id']} - {test['duration']:.1f}s")
    
    # warmup
    model_openai.transcribe(test['wav_path'], language="en", temperature=0)
    
    times = []
    final_text = ""
    for i in range(3):
        t0 = time.time()
        res = model_openai.transcribe(test['wav_path'], language="en", temperature=0)
        times.append(time.time() - t0)
        final_text = res['text']
        
    avg_time = sum(times) / len(times)
    clean_text = re.sub(r'[^\w\s]', '', final_text).lower().strip()
    current_wer = wer(test['ground_truth'], clean_text)
    
    results_ow.append({'id': test['id'], 'time': avg_time, 'wer': current_wer})
    print(f"  avg time: {avg_time:.2f}s | wer: {current_wer:.2%}")

del model_openai
torch.cuda.empty_cache()

In [ ]:
# testing faster-whisper
from faster_whisper import WhisperModel
import time
import torch
from jiwer import wer
import re

compute_type = "float16"
print(f"\n--- faster-whisper ({model_name}, {compute_type}) ---")

t0 = time.time()
model_faster = WhisperModel(model_name, device="cuda", compute_type=compute_type)
load_time_fw = time.time() - t0
print(f"load time: {load_time_fw:.2f}s")

results_fw = []

for test in test_files:
    print(f"\nrunning file {test['id']} - {test['duration']:.1f}s")
    
    # warmup
    list(model_faster.transcribe(test['wav_path'], language="en", temperature=0)[0])
    
    times = []
    final_text = ""
    for i in range(3):
        t0 = time.time()
        segments, info = model_faster.transcribe(test['wav_path'], language="en", temperature=0)
        final_text = " ".join([s.text for s in segments])
        times.append(time.time() - t0)
        
    avg_time = sum(times) / len(times)
    clean_text = re.sub(r'[^\w\s]', '', final_text).lower().strip()
    current_wer = wer(test['ground_truth'], clean_text)
    
    results_fw.append({'id': test['id'], 'time': avg_time, 'wer': current_wer})
    print(f"  avg time: {avg_time:.2f}s | wer: {current_wer:.2%}")

del model_faster
torch.cuda.empty_cache()

In [ ]:
# testing vosk
from vosk import Model, KaldiRecognizer
import time
import wave
import json
from jiwer import wer
import re

print("\n--- vosk (cpu only) ---")

t0 = time.time()
model_vosk = Model("vosk-model-small-en-us-0.15")
load_time_vosk = time.time() - t0
print(f"load time: {load_time_vosk:.2f}s")

results_vosk = []

for test in test_files:
    print(f"\nrunning file {test['id']} - {test['duration']:.1f}s")
    
    times = []
    final_text = ""
    
    for i in range(3):
        t0 = time.time()
        wf = wave.open(test['wav_path'], "rb")
        rec = KaldiRecognizer(model_vosk, wf.getframerate())
        rec.SetWords(True)
        
        transcript_chunks = []
        while True:
            data = wf.readframes(4000)
            if len(data) == 0:
                break
            if rec.AcceptWaveform(data):
                res = json.loads(rec.Result())
                transcript_chunks.append(res.get("text", ""))

        final_res = json.loads(rec.FinalResult())
        transcript_chunks.append(final_res.get("text", ""))
        
        final_text = " ".join(transcript_chunks).strip()
        times.append(time.time() - t0)
        wf.close()
        
    avg_time = sum(times) / len(times)
    clean_text = re.sub(r'[^\w\s]', '', final_text).lower().strip()
    current_wer = wer(test['ground_truth'], clean_text)
    
    results_vosk.append({'id': test['id'], 'time': avg_time, 'wer': current_wer})
    print(f"  avg time: {avg_time:.2f}s | wer: {current_wer:.2%}")

del model_vosk

In [ ]:
# summary
import pandas as pd

print("\n--- summary ---")
print("hardware: t4 gpu (colab)\n")

print("- load times -")
print(f"openai whisper: {load_time_ow:.2f}s")
print(f"faster-whisper: {load_time_fw:.2f}s (loaded {load_time_ow/load_time_fw:.1f}x faster)")
print(f"vosk (cpu):     {load_time_vosk:.2f}s\n")

print("- transcription speeds -")
speed_data = {
    'test id': [t['id'] for t in test_files],
    'length (s)': [f"{t['duration']:.1f}s" for t in test_files],
    'openai whisper': [f"{r['time']:.2f}s" for r in results_ow],
    'faster-whisper': [f"{r['time']:.2f}s" for r in results_fw],
    'vosk': [f"{r['time']:.2f}s" for r in results_vosk],
    'fw speedup vs whisper': [f"{ow['time'] / fw['time']:.1f}x" for ow, fw in zip(results_ow, results_fw)]
}
df_speed = pd.DataFrame(speed_data)
print(df_speed.to_string(index=False))
print("\n")

print("- word error rate (wer) -")
wer_data = {
    'test id': [t['id'] for t in test_files],
    'openai whisper': [f"{r['wer']:.2%}" for r in results_ow],
    'faster-whisper': [f"{r['wer']:.2%}" for r in results_fw],
    'vosk': [f"{r['wer']:.2%}" for r in results_vosk]
}
df_wer = pd.DataFrame(wer_data)
print(df_wer.to_string(index=False))

avg_speedup = sum([ow['time'] / fw['time'] for ow, fw in zip(results_ow, results_fw)]) / len(results_ow)
print(f"\navg faster-whisper speedup: {avg_speedup:.1f}x")